## GPT prompting: n80 examples, third filter

### requires python >= 3.10

### takes only the "no" answers from filter 2 result and runs through all three prompts and saves each result into individual file

In [1]:
import pandas as pd
from tqdm import tqdm
import openai 
import os
from openai import AzureOpenAI
import configparser
import json
import csv
import sys

In [2]:
sys.path.append("../../../")
from common_code.gpt_utils import *
from common_code.gpt_reply_formats import *

In [3]:
from prompts.semantic_categories.v03.prompt import (
    ALIVE_SYSTEM_PROMPT, ALIVE_FEW_SHOTS_STR, ALIVE_FEW_SHOTS, 
    ABSTRACT_SYSTEM_PROMPT, ABSTRACT_FEW_SHOTS_STR, ABSTRACT_FEW_SHOTS, 
    TIME_SYSTEM_PROMPT, TIME_FEW_SHOTS_STR, TIME_FEW_SHOTS
)

from prompts.semantic_categories.v04.prompt import (
    EVENT_SYSTEM_PROMPT, EVENT_FEW_SHOTS_STR, EVENT_FEW_SHOTS, 
    ORG_SYSTEM_PROMPT, ORG_FEW_SHOTS_STR, ORG_FEW_SHOTS
)

In [4]:
pd.set_option('display.max_colwidth', None)
pd.set_option("display.show_dimensions", True)

In [63]:
RESULTS_DIR = "../../results/"

# algse 10k lause joaks
#EXAMPLE_FILE = "n80_examples_large_v01/gpt_v02/" + "gpt_10K_b10_run01.csv"
#GPT_ANSWER_FILE_ALIVE = "n80_examples_large_v01/gpt_v03/"+ "gpt_10K_b10_run01_no_is_alive.csv"
#GPT_ANSWER_FILE_ABSTRACT = "n80_examples_large_v01/gpt_v03/"+ "gpt_10K_b10_run01_no_is_abstract.csv"
#GPT_ANSWER_FILE_TIME = "n80_examples_large_v01/gpt_v03/"+ "gpt_10K_b10_run01_no_is_timex.csv"
#GPT_FILTERED_FILE = "n80_examples_large_v01/gpt_v03/"+ "gpt_10K_b10_run01_no_filtered.csv"

EXAMPLE_FILE = RESULTS_DIR + "n80_examples_large_v02/gpt_v02/" + "gpt_b10_run01.csv"

GPT_ANSWER_FILE_ALIVE = RESULTS_DIR + "n80_examples_large_v02/gpt_v03/"+ "gpt_b10_run01_no_is_alive.csv"
GPT_ANSWER_FILE_ABSTRACT = RESULTS_DIR + "n80_examples_large_v02/gpt_v03/"+ "gpt_b10_run01_no_is_abstract.csv"
GPT_ANSWER_FILE_TIME = RESULTS_DIR + "n80_examples_large_v02/gpt_v03/"+ "gpt_b10_run01_no_is_timex.csv"
GPT_ANSWER_FILE_ORG = RESULTS_DIR + "n80_examples_large_v02/gpt_v03/"+ "gpt_b10_run01_no_is_org.csv"
GPT_ANSWER_FILE_EVENT = RESULTS_DIR + "n80_examples_large_v02/gpt_v03/"+ "gpt_b10_run01_no_is_event.csv"


# fail ainult "no" vastustega (alive+abstract+time kokku panduna)
GPT_FILTERED_FILE = RESULTS_DIR + "n80_examples_large_v02/gpt_v03/"+ "gpt_b10_run01_no_filtered.csv"

# fail kõigi vastustega (kõik laused, tekitatud uus classification3 veerg)
GPT_ANSWER_FILE = RESULTS_DIR + "n80_examples_large_v02/gpt_v03/"+ "gpt_b10_run01.csv"


# väike sample fail 100 näitega
GPT_ANSWER_FILE_SAMP = RESULTS_DIR + "n80_examples_large_v02/gpt_v03/"+ "gpt_b10_run01_sample.csv"



CONF_FILE = "../../../../v04_verb-case_pattern/minu_code/azure.ini"


# OSA I : Andmed


## testimise põhjusel on kasutusel vana 10k v1 andmefail, et tulemusi saaks võrrelda

In [6]:
df = pd.read_csv(EXAMPLE_FILE, encoding="utf-8",  sep=",")

In [7]:
# võtta need, mille puhul gpt ütles "no"

spatial_obl_ex = df[df["classification2"]=="no"]

In [8]:
spatial_obl_ex

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation,classification2,explanation2
5,10060758,16137721,6,jõudma,tagasi,el,lisaraha,lisarahast,"200 miljonit ettevõtete kätte jäävast lisarahast jõuab riigile tagasi üksikisiku tulumaksuna , käibemaksuna ja aktsiisidena .",NaN,NaN,NaN,no,"The phrase 'lisarahast' was classified as not a location because it refers to additional resources or money, not a physical place.",no,The phrase 'lisarahast' was classified as 'no' because it refers to the source of extra money and not a location.
9,15500381,24205992,12,panema,NaN,adit,hooldeprojekt,hooldeprojekti,""" Vajaduse korral panevad Veerpalu ja loodetavasti õed Šmigunid isiklikku raha hooldeprojekti , sest kehvade suuskadega kaugele ei sõida .",NaN,NaN,NaN,no,"The phrase 'hooldeprojekti' was classified as not a location because it refers to a project or initiative, not a physical place.",no,The phrase 'hooldeprojekti' was classified as 'no' because it indicates a target or purpose rather than a place.
12,11302806,18113893,34,olema,NaN,ill,eelarve,eelarvesse,"Tallinna rahandusküsimustega tegelev abilinnapea Ants Leemets kinnitas esmaspäeva õhtul oma kabinetis “ Postimehele ” selge sõnaga , et vähemalt selleks aastaks on Nomura laenu tagasimaksmiseks linnal raha juba olemas , s.o sisse planeeritud eelarvesse , mille volikogu nii või teisiti peagi vastu võtab .",NaN,NaN,NaN,no,The word 'eelarvesse' refers to a budgetary concept and not a physical location.,no,"The phrase 'eelarvesse' refers to a budget or plan and does not indicate a specific location, so it is not an adverbial of place."
13,9623819,15445097,16,hakkama,NaN,ill,sisu,sisusse,"See oli aeg , kus raamatuhuviline poest kõik ilmunud uudiskirjanduse koju tassis ja alles siis sisusse hakkas süvenema .",NaN,NaN,NaN,no,"The word 'sisusse' refers to content or substance, not a specific location.",no,"The phrase 'sisusse' refers to content or substance, not a physical location, so it is not an adverbial of place."
14,14293818,22692055,3,laskma,NaN,adit,Narusk,Naruski,Mae laskis Naruski üheksasekundilisse eduseisu ja alustas siis karmi tagaajamist .,NaN,NaN,LOC,no,"The word 'Naruski' refers to a person's name, not a location.",no,"The phrase 'Naruski' refers to a person's name, not a location, so it is not an adverbial of place."
15,16252383,25172405,39,minema,ära,el,koosseis,koosseisust,"Kolm näidet , mis ma oskan öelda nende kohta , kes on ise lahkunud ( peale selle on lahkujaid ka seoses struktuuri ümberkorraldamisega , ministeeriumist on ära viidud näiteks haldusbüroo ja tehtud muid niisuguseid asju , mistõttu ministeeriumi koosseisust ära läinud isikuid on rohkem ) : näiteks Tallinna linn on saanud Haridusministeeriumist personalijuhi , kes oli töötanud ministeeriumis 13 aastat ja tahtis vaheldust ; üks daam läheb meil järgmisel nädalal Islandile mehele ; üks daam on läinud arvutiõpetajaks Rocca al Mare kooli .",NaN,NaN,NaN,no,"The phrase 'koosseisust' was classified as not a location because it refers to composition or structure, not a physical place.",no,"The phrase 'koosseisust' refers to a composition or structure rather than a location, so it is not an adverbial of place."
17,11102573,17784543,6,ravima,NaN,in,seis,seisus,"Kaks kuud ravis perenaine armetus seisus koera , kaotamata siiski lootust .",NaN,NaN,NaN,no,"The phrase 'seisus' was classified as not a location because it refers to a condition or state, not a physical place.",no,"The phrase 'seisus' refers to a state or condition, not a physical location, so it is not an adverbial of place."
21,3126437,5021032,3,põrkama,kokku,in,algus,alguses,Viimase ringi alguses põrkas Tobreluts raja lahknemiskohas sloveenlasega kokku .,NaN,NaN,NaN,no,The word 'alguses' refers to a point in time (the beginning) and not a specific location.,no,"The word 'alguses' refers to the beginning of an event, specifying time, not place."
23,2891090,4635435,3,saama,NaN

# OSA II : GPT

## GPT jaoks vajalik

In [15]:
config = configparser.ConfigParser()

status = config.read(CONF_FILE) 
assert status == [CONF_FILE]

API_VERSION = config['azure-configuration']['api_version']
AZURE_ENDPOINT = config['azure-configuration']['api_base']
SUBSCRIPTION_KEY = config['azure-configuration']['api_key']
model_name = "gpt-4o" #"GPT-4o-2024-1120 Global"
DEPLOYMENT = config['azure-configuration']['deployment_id']

In [16]:
client = AzureOpenAI(
    api_version=API_VERSION,
    azure_endpoint=AZURE_ENDPOINT,
    api_key=SUBSCRIPTION_KEY,
)

## Andmete söötmine

In [17]:

def classify_batch(my_batch, few_shots, system_prompt, client, deployment):
    """Gets a yes/no answer for a batch of sentences and phrases. 
    """
    #print("classify", len(my_batch))
    max_att = 1
    attempt = 0
    while attempt < max_att:
        attempt += 1
        user_payload = {
            "few_shots": few_shots,
            "batch": my_batch
        }
    
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content":  json.dumps(user_payload, ensure_ascii=False)}
        ]

        #return None, None
        response = client.chat.completions.create(
            model=deployment,
            messages=messages,
            temperature=0, # absoluutselt min väljund 
        )

        raw_output = response.choices[0].message.content.strip()

        try:
            data = json.loads(raw_output)

            if len(data) != len(my_batch):
                raise ValueError(f"Väljundis ei ole õige arv vastuseid. Peaks olema {len(batch)} aga on {len(data)}.")
                
            elif len(data) == len(my_batch):
                for item in data:
                    ClassificationDict(**item)

            return response, raw_output

        except (ValidationError, json.JSONDecodeError, ValueError) as e:
            #print(f"Attempt {attempt} failed. Retrying batch...")
            print(f"Error: {e}")
            #print(f"Raw output: {raw_output[:500]}...")  # preview first 500 chars
            time.sleep(1)  # small delay before retry

    print(f"Batch failed after {max_att} attempts.")
    # isegi kui ei saanud kõike kätte siis saab pärast äkki käsitsi midagi juurde panna
    return response, raw_output


# ALIVE

## NB! muuda max_allowed_tok kui vaja

In [20]:
#df = spatial_obl_ex.sample(frac=1)#.reset_index(drop=True)
df = spatial_obl_ex

In [22]:
results = []
results2 = []
responses = []

used_tokens = 0

# kui palju lauseid gtp-le korraga anda
bs = 10
batch_start_index = 0
batch_cnt = 0

# kui palju on max lubatud tokenid, ehk peale mis tokenite arvu peaks peatama, et üle limiidi ei läheks
max_allowed_tok = 30000

rows = df.to_dict(orient="records")
# kui tahta kõiki näiteid anda gpt-le
for df_batch in tqdm(chunk_data(rows, size=bs)):
    result_yesno = []
    
    batch = []
    for ex in df_batch:
        batch.append( json.dumps({"l": ex["sentence"], "c": ex["form"]}, ensure_ascii=False))

    batch_cnt += 1
    # esmalt klassifitseeri
    response, result = classify_batch(batch, ALIVE_FEW_SHOTS_STR, ALIVE_SYSTEM_PROMPT, client, DEPLOYMENT)
    result_yesno = json.loads(result)
    results += result_yesno
    results2.append(result_yesno)
    responses.append(response)
    used_tokens += response.usage.total_tokens

    batch_start_index += len(batch)

    if used_tokens >= max_allowed_tok:
        print(f"Tehtud on {batch_cnt} batchi ehk {batch_cnt*bs} lauset")
        break    

2it [00:01,  1.29it/s]


In [23]:
used_tokens # 12 lauset batch 10-> 1791 tokenit

1791

In [24]:
len(results)

12

## andmed tabelisse 


In [25]:
if len(results) == len(df):
    df["is_alive"] = [r["a"] for r in results]

In [26]:
df.to_csv(GPT_ANSWER_FILE_ALIVE, encoding="utf-8", index = False, sep=",", quoting=csv.QUOTE_MINIMAL)

# ABSTRACT

In [27]:
df = spatial_obl_ex

## NB! muuda max_allowed_tok kui vaja

In [28]:
results = []
results2 = []
responses = []

used_tokens = 0

# kui palju lauseid gtp-le korraga anda
bs = 10
batch_start_index = 0
batch_cnt = 0
# kui palju on max lubatud tokenid, ehk peale mis tokenite arvu peaks peatama, et üle limiidi ei läheks
max_allowed_tok = 30000

rows = df.to_dict(orient="records")
# kui tahta kõiki näiteid anda gpt-le
for df_batch in tqdm(chunk_data(rows, size=bs)):
    result_yesno = []
    
    batch = []
    for ex in df_batch:
        batch.append(json.dumps({"l": ex["sentence"], "c": ex["form"]}, ensure_ascii=False))

    batch_cnt += 1
    # esmalt klassifitseeri
    response, result = classify_batch(batch, ABSTRACT_FEW_SHOTS_STR, ABSTRACT_SYSTEM_PROMPT, client, DEPLOYMENT)
    result_yesno = json.loads(result)
    results += result_yesno
    results2.append(result_yesno)
    responses.append(response)
    used_tokens += response.usage.total_tokens

    batch_start_index += len(batch)

    if used_tokens >= max_allowed_tok:
        print(f"Tehtud on {batch_cnt} batchi ehk {batch_cnt*bs} lauset")
        break    

2it [00:01,  1.33it/s]


In [29]:
if len(results) == len(df):
    df["is_abstract"] = [r["a"] for r in results]

In [30]:
df.to_csv(GPT_ANSWER_FILE_ABSTRACT, encoding="utf-8", index = False, sep=",", quoting=csv.QUOTE_MINIMAL)

# TIMEX

In [31]:
df = spatial_obl_ex

## NB! muuda max_allowed_tok kui vaja

In [32]:
results = []
results2 = []
responses = []

used_tokens = 0

# kui palju lauseid gtp-le korraga anda
bs = 10
batch_start_index = 0
batch_cnt = 0
# kui palju on max lubatud tokenid, ehk peale mis tokenite arvu peaks peatama, et üle limiidi ei läheks
max_allowed_tok = 30000

rows = df.to_dict(orient="records")
# kui tahta kõiki näiteid anda gpt-le
for df_batch in tqdm(chunk_data(rows, size=bs)):
    result_yesno = []
    
    batch = []
    for ex in df_batch:
        batch.append(json.dumps({"l": ex["sentence"], "c": ex["form"]}, ensure_ascii=False))

    batch_cnt += 1
    # esmalt klassifitseeri
    response, result = classify_batch(batch, TIME_FEW_SHOTS_STR, TIME_SYSTEM_PROMPT, client, DEPLOYMENT)
    result_yesno = json.loads(result)
    results += result_yesno
    results2.append(result_yesno)
    responses.append(response)
    used_tokens += response.usage.total_tokens

    batch_start_index += len(batch)

    if used_tokens >= max_allowed_tok:
        print(f"Tehtud on {batch_cnt} batchi ehk {batch_cnt*bs} lauset")
        break    

2it [00:01,  1.49it/s]


In [33]:
if len(results) == len(df):
    df["is_time"] = [r["a"] for r in results]

In [34]:
df.to_csv(GPT_ANSWER_FILE_TIME, encoding="utf-8", index = False, sep=",", quoting=csv.QUOTE_MINIMAL)

# EVENT

In [20]:
df = pd.read_csv(GPT_FILTERED_FILE, encoding="utf-8", sep=",")

In [24]:
df_2 = df[(df["is_time"]=="no") & (df["is_abstract"]=="no") & (df["is_alive"]=="no")].copy()

## NB! muuda max_allowed_tok kui vaja

In [26]:
results = []
results2 = []
responses = []

used_tokens = 0

# kui palju lauseid gtp-le korraga anda
bs = 10
batch_start_index = 0
batch_cnt = 0
# kui palju on max lubatud tokenid, ehk peale mis tokenite arvu peaks peatama, et üle limiidi ei läheks
max_allowed_tok = 1500000

rows = df_2.to_dict(orient="records")
# kui tahta kõiki näiteid anda gpt-le
for df_batch in tqdm(chunk_data(rows, size=bs)):
    result_yesno = []
    
    batch = []
    for ex in df_batch:
        batch.append(json.dumps({"l": ex["sentence"], "c": ex["form"]}, ensure_ascii=False))

    batch_cnt += 1
    # esmalt klassifitseeri
    response, result = classify_batch(batch, EVENT_FEW_SHOTS_STR, EVENT_SYSTEM_PROMPT, client, DEPLOYMENT)
    result_yesno = json.loads(result)
    results += result_yesno
    results2.append(result_yesno)
    responses.append(response)
    used_tokens += response.usage.total_tokens

    batch_start_index += len(batch)

    if used_tokens >= max_allowed_tok:
        print(f"Tehtud on {batch_cnt} batchi ehk {batch_cnt*bs} lauset")
        break    

74it [00:51,  1.43it/s]


In [27]:
used_tokens # 10 lauset batch 10-> 1600 tokenit, 74 batchi x 10 lauset -> 108,307 tokenit

108307

In [28]:
len(results)

733

In [30]:
if len(results) == len(df_2):
    df_2["is_event"] = [r["a"] for r in results]

In [31]:
df_2.to_csv(GPT_ANSWER_FILE_EVENT, encoding="utf-8", index = False, sep=",", quoting=csv.QUOTE_MINIMAL)

# ORG

In [33]:
df_4 = df_2[df_2["is_event"]=="no"].copy()

In [35]:
results = []
results2 = []
responses = []

used_tokens = 0

# kui palju lauseid gtp-le korraga anda
bs = 10
batch_start_index = 0
batch_cnt = 0
# kui palju on max lubatud tokenid, ehk peale mis tokenite arvu peaks peatama, et üle limiidi ei läheks
max_allowed_tok = 900000

rows = df_4.to_dict(orient="records")
# kui tahta kõiki näiteid anda gpt-le
for df_batch in tqdm(chunk_data(rows, size=bs)):
    result_yesno = []
    
    batch = []
    for ex in df_batch:
        batch.append(json.dumps({"l": ex["sentence"], "c": ex["form"]}, ensure_ascii=False))

    batch_cnt += 1
    # esmalt klassifitseeri
    response, result = classify_batch(batch, ORG_FEW_SHOTS_STR, ORG_SYSTEM_PROMPT, client, DEPLOYMENT)
    result_yesno = json.loads(result)
    results += result_yesno
    results2.append(result_yesno)
    responses.append(response)
    used_tokens += response.usage.total_tokens

    batch_start_index += len(batch)

    if used_tokens >= max_allowed_tok:
        print(f"Tehtud on {batch_cnt} batchi ehk {batch_cnt*bs} lauset")
        break    

68it [00:42,  1.59it/s]


In [36]:
used_tokens

83993

In [37]:
len(results)

673

In [38]:
if len(results) == len(df_4):
    df_4["is_org"] = [r["a"] for r in results]

In [39]:
df_4.to_csv(GPT_ANSWER_FILE_ORG, encoding="utf-8", index = False, sep=",", quoting=csv.QUOTE_MINIMAL)

# Kokku filtreeritud fail

In [43]:
fname1 = GPT_ANSWER_FILE_ALIVE
fname2 = GPT_ANSWER_FILE_EVENT
fname3 = GPT_ANSWER_FILE_TIME
fname4 = GPT_ANSWER_FILE_ORG
fname5 = GPT_ANSWER_FILE_ABSTRACT

df1 = pd.read_csv(fname1, encoding="utf-8",  sep=",")
df2 = pd.read_csv(fname2, encoding="utf-8",  sep=",")
df3 = pd.read_csv(fname3, encoding="utf-8",  sep=",")
df4 = pd.read_csv(fname4, encoding="utf-8",  sep=",")
df5 = pd.read_csv(fname5, encoding="utf-8",  sep=",")

key_cols = ['sentence_id','head_id', "verb", "verb_compound", "morph_case", "form"]

df2_selected = df2[key_cols + ["is_event"]].copy()
df3_selected = df3[key_cols + ["is_time"]].copy()
df4_selected = df4[key_cols + ["is_org"]].copy()
df5_selected = df5[key_cols + ["is_abstract"]].copy()

df1['verb_compound'] = df1['verb_compound'].astype('string').str.strip()
df2_selected['verb_compound'] = df2_selected['verb_compound'].astype('string').str.strip()
df3_selected['verb_compound'] = df3_selected['verb_compound'].astype('string').str.strip()
df4_selected['verb_compound'] = df4_selected['verb_compound'].astype('string').str.strip()
df5_selected['verb_compound'] = df5_selected['verb_compound'].astype('string').str.strip()

# Merge df1 with df2_selected etc
merged_df = df1.merge(df2_selected, on=key_cols, how='left')

merged_df2 = merged_df.merge(df3_selected, on=key_cols, how='left')

merged_df3 = merged_df2.merge(df4_selected, on=key_cols, how='left')

final_df = merged_df3.merge(df5_selected, on=key_cols, how='left')

cols = ['is_time', 'is_event', 'is_alive', 'is_org', 'is_abstract']

final_df[cols] = final_df[cols].fillna('no')
final_df['verb_compound'] = final_df['verb_compound'].fillna('')

In [45]:
final_df.to_csv(GPT_FILTERED_FILE, encoding="utf-8", index = False, sep=",", quoting=csv.QUOTE_MINIMAL)

## Kokku kõikide varasemate klassifikatsioonidega + classification3 loomine

### seda osa saab korrata ilma gpt osa uuesti tegemata

In [56]:
df1 = pd.read_csv(EXAMPLE_FILE, encoding="utf-8", sep=",")
df2 = pd.read_csv(GPT_FILTERED_FILE, encoding="utf-8", sep=",")

In [57]:
cols = ["head_id", "form", "verb", "verb_compound", "morph_case","sentence_id", "is_time", "is_alive", "is_abstract", "is_org", "is_event"]
df3 = df2[cols]

In [58]:
filter3 = pd.merge(df1, df3, on=["head_id", "form", "verb", "verb_compound", "morph_case", "sentence_id"], how='left')
filter3

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,classification,...,classification2,explanation2,timex_tag,ekilex_tag,ner_tag,is_time,is_alive,is_abstract,is_org,is_event
0,2792027,4477403,1,sadama,maha,in,Tallinn,Tallinnas,Tallinnas laupäeval maha sadanud lumi lõi ilmajaama andmeil kümne aasta rekordi .,yes,...,yes,"The phrase 'Tallinnas' specifies a location (in Tallinn), so it is adverbial of place.",NaN,location,LOC,NaN,NaN,NaN,NaN,NaN
1,2149636,3433963,8,ringlema,NaN,in,piletiäri,piletiäris,"Lihtsad arvutused näitavad , et Tallinna põrandaaluses piletiäris ringlevad summad on tohutud .",no,...,yes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,12372585,19808269,6,sööma,NaN,in,fuajee,fuajees,"Etenduse vaheajal sõid lapsed teatri fuajees puuvilju ning mängis ansambel "" Üks lust "" .",yes,...,yes,NaN,NaN,location,NaN,NaN,NaN,NaN,NaN,NaN
3,6427967,10334047,1,kiirustama,NaN,adit,õnnetuspaik,Õnnetuspaika,Õnnetuspaika kiirustanud Soome ja Eesti päästekopterid meest enne pimeduse saabumist ei leidnud .,yes,...,yes,NaN,NaN,location,NaN,NaN,NaN,NaN,NaN,NaN
4,13574774,21705880,5,lubama,NaN,ill,Vilnius,Vilniusesse,"SK Polaris ei lubanud Vilniusesse Jaanus Liivakut , nii tugevdavad Kalevit Valmo Kriisa Nybitist ja esmakordselt Kristo Reinumäe Canon-Eesti noortemeeskonnast .",yes,...,yes,NaN,NaN,location,LOC,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,5763409,9245685,6,voolama,välja,el,teokarp,teokarbist,"Ka vetejumala jalgade juures olevast teokarbist voolab välja vesi , mis valgub mööda kaskaadi astmeid allapoole .",no,...,yes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9996,16806790,25865186,1,varisema,kokku,in,Thbilisi,Thbilisis,Thbilisis varises kokku kaks elamut .,yes,...,yes,NaN,NaN,location,LOC,NaN,NaN,NaN,NaN,NaN
9997,7695875,12338914,10,laskma,NaN,ill,nimekiri,nimekirjadesse,"Ilma arstiabita ei jää ka need , kes ennast nimekirjadesse ei lase kanda , kinnitab Hillar Kalda .",no,...,yes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9998,6260170,10056376,8,toimuma,NaN,ill,linnus,linnusesse,"20. augusti õhtul toimub rongkäik Rakvere spordihallist linnusesse , kus kella 23ni toimub rahvapidu .",yes,...,yes,NaN,NaN,location,NaN,NaN,NaN,NaN,NaN,NaN


In [59]:
# uus classification3
# muuta vastavalt vajadusele
"""
def new_class(row):
    
    # kui filter2 tulemus = "yes" -> jääb "yes"
    if row["classification2"] == "yes":
        return "yes"
    
    # kui on aeg -> "no"
    if row["is_time"] == "yes":
        return "no"
    
    # oletame, et elus võib olla koht -> "yes"
    if row["is_alive"] == "yes":
        return "yes"
    
    # oletame, et abstraktne asi on koht -> "yes"
    if row["is_abstract"] == "yes":
        return "yes"
    
    # kui oli "no" ja/või alive/time/abstract kõik olid "no"
    else:
        return "no"
    
"""    
    
def new_class(row):
    
    # kui eelmine tulemus = "yes" -> jääb "yes"  -> saame välja visata
    if row["classification"] == "yes":
        return "loc"
    if row["is_abstract"] == "yes":
        return "loc"
    
    # kui on aeg -> "yes" -> saame välja visata
    if row["is_time"] == "yes":
        return "time"
    
    # event -> "yes" -> saame välja visata
    if row["is_event"] == "yes":
        return "event"
    
    # elus -> "yes" -> saame välja visata
    if row["is_alive"] == "yes":
        return "actor"
    
    # org -> "yes" -> saame välja visata
    if row["is_org"] == "yes":
        return "actor"

    
    # kui oli "no", NaN ja/või alive/time/abstract kõik olid "no"
    else:
        return "UNK"

In [60]:
filter3["classification3"] = filter3.apply(new_class, axis=1)

In [61]:
filter3.to_csv(GPT_ANSWER_FILE, encoding="utf-8", index=False, sep=",", quoting=csv.QUOTE_MINIMAL)

In [64]:
filter3_2 = filter3.iloc[:100]

In [65]:
filter3_2.to_csv(GPT_ANSWER_FILE_SAMP, encoding="utf-8", index=False, sep=",", quoting=csv.QUOTE_MINIMAL)